# Rotary Positional Encoding Visualizations

Interactive views of how each RoPE variant transforms a feature vector as a function of position `(x, y)`.

- **2-D rotation methods** (Axial / Learned Axial / Mixed RoPE) — channels grouped into 2-D pairs, each shown on its own circle.
- **3-D rotation methods** (Spherical, Learned Spherical, QuatRo, Spherical QuatRo, Mixed QuatRo) — channels grouped into 3-D triples, each shown on its own sphere.
- **CARE** — 8-D Cl(3) multivector decomposed into grades (scalar, vector, bivector, pseudoscalar).

The original vector is shown in **blue**, the rotated vector in **red**. Use the dropdown to switch methods and the sliders to move `(x, y)`.

In [1]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3d projection)
from ipywidgets import interact, FloatSlider, Dropdown
%matplotlib inline
torch.manual_seed(0)
np.random.seed(0)

In [2]:
from PE_registry import PE_REGISTRY, register_PE
import Positional_Embeddings

SphericalRoPE = PE_REGISTRY["Spherical RoPE"]
QuatRo = PE_REGISTRY["QuatRo"]
MixedRoPE = PE_REGISTRY["Mixed RoPE"]
MixedQuatRo = PE_REGISTRY["Mixed QuatRo"]
SphericalQuatRo = PE_REGISTRY["Spherical QuatRo"]
AxialRoPE = PE_REGISTRY["Axial RoPE"]
CARE_PE = PE_REGISTRY["CARE"]


Triton not available, falling back to pure PyTorch implementation.


In [3]:
# ---------- drawing utilities ----------

def _draw_circle(ax, r, alpha=0.3):
    th = np.linspace(0, 2 * np.pi, 200)
    ax.plot(r * np.cos(th), r * np.sin(th), 'k--', alpha=alpha)

def _draw_sphere(ax, r, alpha=0.10):
    u = np.linspace(0, 2 * np.pi, 30)
    v = np.linspace(0, np.pi, 20)
    xs = r * np.outer(np.cos(u), np.sin(v))
    ys = r * np.outer(np.sin(u), np.sin(v))
    zs = r * np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color='gray', alpha=alpha, linewidth=0.4)

def _arrow_2d(ax, v, color, alpha=1.0):
    ax.annotate('', xy=(v[0].item(), v[1].item()), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2, alpha=alpha))

def _arrow_3d(ax, v, color, alpha=1.0):
    v = v.numpy() if hasattr(v, 'numpy') else np.asarray(v)
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color=color, alpha=alpha,
              arrow_length_ratio=0.15, linewidth=2)

## 2-D rotation methods

Channels are split into 2-D pairs and each is rotated on a circle. 
Axial RoPE here uses `D = 8` (so `D / (2·M) = 2` pairs per axis → 4 circles); 
Mixed RoPE uses `D = 6` (3 circles).

In [4]:
# Fixed inputs (unit-ish vectors)
Z6 = torch.randn(6); Z6 = Z6 / Z6.norm() * 0.9
Z8 = torch.randn(8); Z8 = Z8 / Z8.norm() * 0.9

_methods_2d = {
    'Axial RoPE (fixed freq)':  (AxialRoPE(embedding_dim=8, pos_dim=2), torch.tensor(Z8).view(1,1,1,8)),
    'Mixed RoPE (fixed freq)':   (MixedRoPE(embedding_dim=8, pos_dim=2, heads=1), torch.tensor(Z8).view(1,1,1,8)),
}

def viz_2d(method, pos_x, pos_y):
    m, z0 = _methods_2d[method]
    p = torch.tensor([pos_x, pos_y]).view(1,1,2)
    z_rot = m(z0, p)
    p0 = z0.view(-1, 2)
    pr = z_rot.view(-1, 2)
    n = p0.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(3.0 * n, 3.0))
    if n == 1: axes = [axes]
    for i, ax in enumerate(axes):
        r = float(torch.linalg.norm(p0[i]))
        _draw_circle(ax, r)
        _arrow_2d(ax, p0[i], 'steelblue', alpha=0.5)
        _arrow_2d(ax, pr[i], 'crimson')
        L = max(1.0, r) * 1.3
        ax.set_xlim(-L, L); ax.set_ylim(-L, L)
        ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
        ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)
        ax.set_title(f'pair {i}')
    fig.suptitle(method, y=1.02)
    plt.tight_layout(); plt.show()

interact(viz_2d,
    method=Dropdown(options=list(_methods_2d.keys()), description='method'),
    pos_x=FloatSlider(min=-math.pi, max=math.pi, step=0.05, value=0.0, description='x'),
    pos_y=FloatSlider(min=-math.pi, max=math.pi, step=0.05, value=0.0, description='y'));

/var/folders/y1/3gkl7w6939n628by8m9n0gtm0000gn/T/ipykernel_96021/136383226.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'Axial RoPE (fixed freq)':  (AxialRoPE(embedding_dim=8, pos_dim=2), torch.tensor(Z8).view(1,1,1,8)),
/var/folders/y1/3gkl7w6939n628by8m9n0gtm0000gn/T/ipykernel_96021/136383226.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'Mixed RoPE (fixed freq)':   (MixedRoPE(embedding_dim=8, pos_dim=2, heads=1), torch.tensor(Z8).view(1,1,1,8)),


interactive(children=(Dropdown(description='method', options=('Axial RoPE (fixed freq)', 'Mixed RoPE (fixed fr…

## 3-D rotation methods

Channels are split into 3-D triples and each is rotated on a sphere (`D = 6` → 2 spheres).

In [ ]:
Z6_3d = torch.randn(6).view(1,1,1,6)
Z6_3d = Z6_3d / Z6_3d.norm() * 0.9

_methods_3d = {
    'Spherical RoPE (fixed)':  SphericalRoPE(embedding_dim=6),
    'QuatRo':                   QuatRo(embedding_dim=6, heads=1).eval(),
    'Spherical QuatRo':         SphericalQuatRo(embedding_dim=6, heads=1).eval(),
    'Mixed QuatRo':             MixedQuatRo(embedding_dim=6, heads=1).eval(),
}

def viz_3d(method, pos_x, pos_y):
    m = _methods_3d[method]
    p = torch.tensor([pos_x, pos_y]).view(1,1,2)
    z_rot = m(Z6_3d, p).detach()
    t0, tr = Z6_3d.view(-1, 3), z_rot.view(-1, 3)
    n = t0.shape[0]
    fig = plt.figure(figsize=(4.5 * n, 4.0))
    for i in range(n):
        ax = fig.add_subplot(1, n, i + 1, projection='3d')
        r = float(torch.linalg.norm(t0[i]))
        _draw_sphere(ax, r)
        _arrow_3d(ax, t0[i], 'steelblue', alpha=0.55)
        _arrow_3d(ax, tr[i], 'crimson')
        L = max(1.0, r) * 1.2
        ax.set_xlim(-L, L); ax.set_ylim(-L, L); ax.set_zlim(-L, L)
        ax.set_box_aspect([1, 1, 1])
        ax.set_title(f'triple {i}')
    fig.suptitle(method, y=1.02)
    plt.tight_layout(); plt.show()

interact(viz_3d,
    method=Dropdown(options=list(_methods_3d.keys()), description='method'),
    pos_x=FloatSlider(min=-math.pi, max=math.pi, step=0.05, value=0.0, description='x'),
    pos_y=FloatSlider(min=-math.pi, max=math.pi, step=0.05, value=0.0, description='y'));

interactive(children=(Dropdown(description='method', options=('Spherical RoPE (fixed)', 'QuatRo', 'Spherical Q…

## CARE — 8-D Cl(3) multivector

Layout: `[s, e1, e2, e3, e23, e31, e12, e123]` — one scalar, three vector components, 
three bivector components, one pseudoscalar. The rotor `R` leaves grade 0 and grade 3 invariant 
and rotates the vector and bivector grades (each as a 3-vector under the *same* SO(3) rotation).

Plots: the two invariant grades (left bars), the rotated **vector** (3-D arrow), the rotated **bivector** 
(3-D arrow), and a final bar chart of all 8 components for direct comparison.

In [ ]:
MV0_ = torch.randn(8).view(1,1,1,8); MV0 = MV0 / MV0.norm() * 0.9
CARE = PE_REGISTRY["CARE"]
_care = CARE(embedding_dim=8, heads=1, random_init=True)

def viz_care(pos_x, pos_y):
    p = torch.tensor([pos_x, pos_y]).view(1,1,2)
    mv_r = _care(MV0_, p)
    fig = plt.figure(figsize=(14, 4))
    MV0=MV0_.squeeze()
    mv_r = mv_r.squeeze().detach()

    # invariant grades: scalar + pseudoscalar
    ax_inv = fig.add_subplot(1, 4, 1)
    xs = np.arange(2)
    ax_inv.bar(xs - 0.18, [MV0[0].item(), MV0[7].item()], width=0.36,
               color='steelblue', alpha=0.6, label='orig')
    ax_inv.bar(xs + 0.18, [mv_r[0].item(), mv_r[7].item()], width=0.36,
               color='crimson', label='rotated')
    ax_inv.set_xticks(xs); ax_inv.set_xticklabels(['scalar (g0)', 'pseudo (g3)'])
    ax_inv.axhline(0, color='k', lw=0.5)
    ax_inv.set_title('invariant grades')
    ax_inv.legend(loc='best', fontsize=8)

    # vector (grade 1)
    v0, vr = MV0[1:4], mv_r[1:4]
    ax_v = fig.add_subplot(1, 4, 2, projection='3d')
    _draw_sphere(ax_v, float(torch.linalg.norm(v0)))
    _arrow_3d(ax_v, v0, 'steelblue', alpha=0.55)
    _arrow_3d(ax_v, vr, 'crimson')
    L = max(1.0, float(torch.linalg.norm(v0))) * 1.2
    ax_v.set_xlim(-L, L); ax_v.set_ylim(-L, L); ax_v.set_zlim(-L, L)
    ax_v.set_box_aspect([1, 1, 1])
    ax_v.set_title('vector (grade 1)')

    # bivector (grade 2)
    b0, br = MV0[4:7], mv_r[4:7]
    ax_b = fig.add_subplot(1, 4, 3, projection='3d')
    _draw_sphere(ax_b, float(torch.linalg.norm(b0)))
    _arrow_3d(ax_b, b0, 'steelblue', alpha=0.55)
    _arrow_3d(ax_b, br, 'crimson')
    L = max(1.0, float(torch.linalg.norm(b0))) * 1.2
    ax_b.set_xlim(-L, L); ax_b.set_ylim(-L, L); ax_b.set_zlim(-L, L)
    ax_b.set_box_aspect([1, 1, 1])
    ax_b.set_title('bivector (grade 2)')

    # all 8 components
    ax_all = fig.add_subplot(1, 4, 4)
    names = ['s', 'e1', 'e2', 'e3', 'e23', 'e31', 'e12', 'I']
    xs = np.arange(8)
    ax_all.bar(xs - 0.18, MV0.numpy(), width=0.36, color='steelblue', alpha=0.6, label='orig')
    ax_all.bar(xs + 0.18, mv_r.numpy(), width=0.36, color='crimson', label='rotated')
    ax_all.set_xticks(xs); ax_all.set_xticklabels(names)
    ax_all.axhline(0, color='k', lw=0.5)
    ax_all.set_title('all 8 components')
    ax_all.legend(loc='best', fontsize=8)

    plt.tight_layout(); plt.show()

interact(viz_care,
    pos_x=FloatSlider(min=-math.pi, max=math.pi, step=0.05, value=0.0, description='x'),
    pos_y=FloatSlider(min=-math.pi, max=math.pi, step=0.05, value=0.0, description='y'));

interactive(children=(FloatSlider(value=0.0, description='x', max=3.141592653589793, min=-3.141592653589793, s…